<a href="https://colab.research.google.com/github/shin584/project/blob/1D_models/Cas12a_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

수정사항
- 고도화 : 코드 최적화 및 모델 성능 향상
ex) 새로운 로직 적용 이후 점수 상향 여부 확인
- 베이스 모델과 구조 및 로직 차이를 확인하고 모델간 일관성 유지
  - 모델간 구조가 다를 경우 성능 향상 여부를 확인
  - 성능이 향상되는 방향으로 모델에 로직 적용
- 테스트셋 분리

In [8]:
import os
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 파일 경로 설정
# '내 드라이브'는 Colab에서 '/content/drive/MyDrive'로 인식됩니다.
base_path = '/content/drive/MyDrive/Colab Notebooks/project_CAS/data'
file_name = 'kim_2018_cas12a_dataset.xlsx'
file_path = os.path.join(base_path, file_name)

if os.path.exists(file_path):
    print(f"파일을 찾았습니다! 경로: {file_path}")
else:
    print(f"파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {file_path}")
    print("팁: 폴더명이나 파일명에 띄어쓰기가 있는지, 대소문자가 맞는지 확인해보세요.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
파일을 찾았습니다! 경로: /content/drive/MyDrive/Colab Notebooks/project_CAS/data/kim_2018_cas12a_dataset.xlsx


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, BatchNormalization, Dropout, Bidirectional, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [4]:
# ---------------------------------------------------------
# 1. Cas12a 전용 원-핫 인코딩 (34bp 고정)
# 34bp = 4bp(5'주변염기) + 4bp(pam서열,TTTV) + 23bp(타겟서열) + 3bp(3'주변염)
# ---------------------------------------------------------
def cas12a_one_hot(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    sequence = sequence.upper()
    # 34bp보다 짧으면 에러, 길면 자름
    if len(sequence) < 34: return None
    arr = [mapping.get(base, [0,0,0,0]) for base in sequence[:34]]
    return np.array(arr)

In [5]:
# ---------------------------------------------------------
# 2. Cas12a 데이터 로드
# ---------------------------------------------------------
def load_cas12a_only(path):
    print(f"Cas12a 데이터 파일 로딩 중: {path}")
    df = pd.read_excel(path)

    # 1. 서열 컬럼 찾기
    try:
        seq_col = [c for c in df.columns if '34 bp' in c][0]
    except:
        seq_col = '34 bp synthetic target and target context sequence (4 bp + PAM + 23 bp protospacer + 3 bp)'

    # 2. 점수 컬럼 찾기
    all_cols = df.columns.tolist()

    # [1순위] 우리가 원하는 'Background subtracted(실험군데이터에서 대조군데이터를 뺀 순수 교정 효율)'
    priority_1 = [c for c in all_cols if 'Indel' in c and ('subtracted' in c or 'substracted' in c)]

    # [2순위] 차선책 (Background라는 말 없는 것) 수정사항 : 굳이 차선책 필요없을듯..?
    priority_2 = [c for c in all_cols if 'Indel' in c and 'Background' not in c]

    score_col = None
    if priority_1:
        score_col = priority_1[0] # 1순위
        print("   [성공] 'Background subtracted' 컬럼을 찾았습니다!")
    elif priority_2:
        score_col = priority_2[0] # 없으면 이거라도...
        print("   [주의] 'subtracted' 컬럼이 없어서 일반 Indel 컬럼을 사용합니다.")
    else:
        print("에러: 적절한 점수 컬럼을 찾지 못했습니다.")
        return None, None

    print(f"   최종 선택된 컬럼: '{score_col}'")

    X_list = [] # 타겟서열리스트
    Y_list = [] # 점수(정답)리스트

    for _, row in df.iterrows():
        seq = row[seq_col]
        if not isinstance(seq, str): continue # string 타입 확인

        encoded_seq = cas12a_one_hot(seq)
        if encoded_seq is None: continue

        score = pd.to_numeric(row[score_col], errors='coerce')
        if np.isnan(score): continue

        X_list.append(encoded_seq)
        Y_list.append(score)

    return np.array(X_list), np.array(Y_list)

In [10]:
# ---------------------------------------------------------
# 3. Cas12a 전용 모델 설계
# ---------------------------------------------------------
def build_cas12a_model():
    # 입력: 깔끔하게 34bp (SpCas9 패딩 불필요)
    input_seq = Input(shape=(34, 4), name='In_Seq_34bp')

    # [CNN] 지역적 특징 (Motif)
    x = Conv1D(128, 5, activation='relu', padding='same')(input_seq)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # [Bi-LSTM] 양방향으로 두 개의 LSTM사용
    #   - cas9과 pam 위치가 완전히 다름. 5` 방향에 PAM 서열 존재하므로 소실되지않게 하기 위함.
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = Dropout(0.3)(x)

    # [Dense] 회귀 예측
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='linear', name='Output_Efficiency')(x)

    model = Model(inputs=input_seq, outputs=output)
    return model

In [11]:
# =========================================================
# [실행 파트]
# =========================================================


X, Y = load_cas12a_only(file_path)

if X is not None:
    # 셔플 (필수) - overfitting 방지용
    # 1. 데이터의 순서나 패턴을 외워버리지 못하도록 방지
    # 2. 특정 배치에 값이 몰리는것을 방지
    indices = np.arange(len(X)) # 타겟서열 갯수만큼 배열생성(0~len(X)의 값)
    np.random.shuffle(indices)
    # 행 순서를 무작위로 섞음
    X = X[indices]
    Y = Y[indices]

    print(f"데이터 준비 완료: {len(X)}개 샘플")

    # 2. 모델 학습
    model = build_cas12a_model()
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])

    # 콜백 (과적합 방지)
    callbacks = [
        # 3회 연속으로 최고점수를 달성하지 못하면 학습률 감소
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1),
        # 학습률 감소 이후로 10회 연속으로 최고점수를 달성하지 못하면 학습중단
        # 중단 이후 이전의 가장 점수가 높았던 상태로 모델을 복
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]

    print("\nCas12a 전용 모델 학습 시작!")
    history = model.fit(
        X, Y,
        batch_size=64,
        epochs=50,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1
    )

    # 3. 저장
    model.save('Cas12a_Only.keras')
    print("\n모델 저장 완료: Cas12a_Only.keras")

Cas12a 데이터 파일 로딩 중: /content/drive/MyDrive/Colab Notebooks/project_CAS/data/kim_2018_cas12a_dataset.xlsx
   [성공] 'Background subtracted' 컬럼을 찾았습니다!
   최종 선택된 컬럼: 'Indel freqeuncy
(Background substracted, %)'
데이터 준비 완료: 15000개 샘플

Cas12a 전용 모델 학습 시작!
Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 26s 108ms/step - loss: 1222.5164 - mae: 29.1961 - val_loss: 1138.8015 - val_mae: 28.6449 - learning_rate: 0.0010
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - loss: 799.9644 - mae: 23.7027 - val_loss: 959.5156 - val_mae: 24.8554 - learning_rate: 0.0010
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - loss: 616.7980 - mae: 19.6747 - val_loss: 562.2266 - val_mae: 17.8621 - learning_rate: 0.0010
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - loss: 554.0143 - mae: 18.3989 - val_loss: 469.8579 - val_mae: 16.7820 - learning_rate: 0.0010
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - loss: 524.8493 - mae: 17.7143 - val_loss: 468.3499 - val_mae: 16.8347 - learning_rate: 0.00

In [12]:

# 모델 저장 경로 설정
# Cas12a_Only.keras는 학습 후 자동으로 저장된 모델 파일명입니다.
model_save_path = os.path.join(base_path, 'Cas12a_Only.keras')

# 현재 학습된 모델을 Google Drive에 저장
model.save(model_save_path)
print(f"💾 모델이 Google Drive에 저장되었습니다: {model_save_path}")

💾 모델이 Google Drive에 저장되었습니다: /content/drive/MyDrive/Colab Notebooks/project_CAS/data/Cas12a_Only.keras
